In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

In [2]:
# 1. Generator: turns noise into fake images
class Generator(nn.Module):
    def __init__(self, input_size=100, output_size=784):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, output_size),
            nn.Tanh()  # Outputs are between -1 and 1
        )

    def forward(self, z):
        return self.model(z)

In [3]:
# 2. Discriminator: classifies real vs fake images
class Discriminator(nn.Module):
    def __init__(self, input_size=784):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()  # Outputs probability
        )

    def forward(self, x):
        return self.model(x)

In [4]:
# 3. Noise generator
def generate_noise(batch_size, z_dim):
    return torch.randn(batch_size, z_dim)

In [5]:
# 4. Load MNIST data (28x28 images)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])
dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 481kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.7MB/s]


In [6]:
# 5. Initialize models, optimizers, and loss
z_dim = 100
G = Generator(z_dim)
D = Discriminator()
criterion = nn.BCELoss()
G_opt = optim.Adam(G.parameters(), lr=0.0002)
D_opt = optim.Adam(D.parameters(), lr=0.0002)

In [7]:
# 6. Training loop
epochs = 5
fixed_noise = generate_noise(16, z_dim)

os.makedirs("gan_outputs", exist_ok=True)

for epoch in range(epochs):
    for real_images, _ in loader:
        batch_size = real_images.size(0)
        real_images = real_images.view(batch_size, -1)

        # Labels
        real_labels = torch.ones(batch_size, 1)
        fake_labels = torch.zeros(batch_size, 1)

        # ---- Train Discriminator ----
        noise = generate_noise(batch_size, z_dim)
        fake_images = G(noise).detach()
        D_loss_real = criterion(D(real_images), real_labels)
        D_loss_fake = criterion(D(fake_images), fake_labels)
        D_loss = D_loss_real + D_loss_fake

        D.zero_grad()
        D_loss.backward()
        D_opt.step()

        # ---- Train Generator ----
        noise = generate_noise(batch_size, z_dim)
        fake_images = G(noise)
        G_loss = criterion(D(fake_images), real_labels)  # Fool the discriminator

        G.zero_grad()
        G_loss.backward()
        G_opt.step()

    # Save sample images after each epoch
    with torch.no_grad():
        samples = G(fixed_noise).view(-1, 1, 28, 28)
        grid = utils.make_grid(samples, nrow=4, normalize=True)
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis('off')
        plt.title(f"Epoch {epoch+1}")
        plt.savefig(f"gan_outputs/epoch_{epoch+1}.png")
        plt.close()

In [9]:
from google.colab import files
files.download("gan_outputs/epoch_5.png")  # Replace with the file you want

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>